In [1]:
import logging
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.datasets import make_regression

from mla.linear_models import LinearRegression, LogisticRegression
from mla.metrics.metrics import mean_squared_error, accuracy



In [2]:
# change to DEBUG to see convergence 
logging.basicConfig(level=logging.ERROR)

In [4]:

def regression():
    x, y = make_regression(
        n_samples=10000,
        n_features=100,
        n_informative=75,
        n_targets=1,
        noise=0.05,
        random_state=1111,
        bias=0.5,
    )

    X_train, X_test, y_train, y_test = train_test_split(
        x, y, test_size=0.25, random_state=1111
    )

    model = LinearRegression(lr=0.01, max_iters=2000, penalty="l2", C=0.03)
    model.fit(X_train, y_train)

    predictions = np.array(model.predict(X_test), dtype=float)
    y_test = np.array(y_test, dtype=float)

    print("regression mse", mean_squared_error(y_test, predictions))

Metrics:

In [11]:

def confusion_matrix_manual(y_true, y_pred):
    tp = fp = tn = fn = 0

    for yt, yp in zip(y_true, y_pred):
        if yt == 1 and yp == 1:
            tp += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 0 and yp == 0:
            tn += 1
        elif yt == 1 and yp == 0:
            fn += 1

    return tp, fp, tn, fn

def precision(tp, fp):
    return tp / (tp + fp) if (tp + fp) != 0 else 0

def recall(tp, fn):
    return tp / (tp + fn) if (tp + fn) != 0 else 0

def f1(p, r):
    return 2 * p * r / (p + r) if (p + r) != 0 else 0

def balanced_accuracy(tp, fp, tn, fn):
    tpr = tp / (tp + fn) if (tp + fn) != 0 else 0
    tnr = tn / (tn + fp) if (tn + fp) != 0 else 0
    return (tpr + tnr) / 2

encoding labels

In [12]:

def encode_labels(series):
    """
    handles:
      - "P" / "N" 
      - 1  / -1   
      - 1  /  0   
    """
    s = series.astype(str).str.strip().str.upper()
    unique = set(s.unique())

    if unique == {"P", "N"}:
        return s.map({"P": 1, "N": 0}).astype(int)

    if unique == {"1", "-1"}:
        return s.map({"1": 1, "-1": 0}).astype(int)

    if unique == {"1", "0"}:
        return s.map({"1": 1, "0": 0}).astype(int)

    raise ValueError(f"Unknown label format: {unique}. Add a mapping above.")



classification

In [13]:

def classification(file_path, target_col):

    df = pd.read_csv(file_path)
    y = encode_labels(df[target_col])

    X = df.drop(columns=[target_col])
    X = X.replace(["?", "NA", "N/A", "null", "", "unknown"], np.nan)
    X = X.dropna()
    y = y.loc[X.index]

    X = pd.get_dummies(X)
    X = X.to_numpy(dtype=float)
    y = y.to_numpy(dtype=int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.1, random_state=1111
    )

    model = LogisticRegression(lr=0.01, max_iters=500, penalty="l1", C=0.01)
    model.fit(X_train, y_train)

    predictions = (np.array(model.predict(X_test), dtype=float) >= 0.5).astype(int)

    tp, fp, tn, fn = confusion_matrix_manual(y_test, predictions)
    p = precision(tp, fp)
    r = recall(tp, fn)

    print(f"\n{'='*48}")
    print(f"  {file_path}")
    print(f"{'='*48}")
    print(f"  Accuracy         : {accuracy(y_test, predictions):.4f}")
    print(f"  Precision        : {p:.4f}")
    print(f"  Recall           : {r:.4f}")
    print(f"  F1               : {f1(p, r):.4f}")
    print(f"  Balanced Accuracy: {balanced_accuracy(tp, fp, tn, fn):.4f}")
    print(f"{'='*48}")

In [14]:

if __name__ == "__main__":
    regression()
    classification("class_imbalance/dataset_867_visualizing_livestock.csv", "binaryClass")
    classification("class_imbalance/dataset_765_analcatdata_apnea2.csv", "binaryClass")
    classification("class_imbalance/dataset_311_oil_spill.csv", "class")

regression mse 67.15616364304071

  class_imbalance/dataset_867_visualizing_livestock.csv
  Accuracy         : 0.9231
  Precision        : 0.9167
  Recall           : 1.0000
  F1               : 0.9565
  Balanced Accuracy: 0.7500

  class_imbalance/dataset_765_analcatdata_apnea2.csv
  Accuracy         : 0.8333
  Precision        : 0.8333
  Recall           : 1.0000
  F1               : 0.9091
  Balanced Accuracy: 0.5000

  class_imbalance/dataset_311_oil_spill.csv
  Accuracy         : 0.0745
  Precision        : 0.0745
  Recall           : 1.0000
  F1               : 0.1386
  Balanced Accuracy: 0.5000


/Users/olapacia/miniconda3/envs/ml_env/lib/python3.11/site-packages/autograd/tracer.py:54: RuntimeWarning: overflow encountered in cosh
  return f_raw(*args, **kwargs)
